# Quickstart: Using NexaMem with Azure Redis Cache

This notebook demonstrates how to configure and use NexaMem with Azure Redis Cache for chatbot history management. It uses the local development version of NexaMem.

Import NexaMem

In [10]:
import sys
import os
import subprocess

# Add the src directory to the Python path
# This is so we can import the local nexamem package
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

# Install the redis package
subprocess.run([sys.executable, "-m", "pip", "install", "redis"], check=True)

CompletedProcess(args=['c:\\Users\\chadada\\source\\repos\\nexamem\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'redis'], returncode=0)

Configure and create history

In [11]:
from nexamem.config import initialize
from nexamem.history import ChatHistory
import json

# 1. Configure NexaMem to use Azure Redis Cache
# Replace with your Azure Redis Cache endpoint and primary key
redis_config = {
    "host": "[your-hostname].redis.cache.windows.net",
    "port": 6380,
    "password": "[Your-Primary-Key]",
    "ssl": True
}

initialize({
    "history_storage": "redis",
    "redis_config": redis_config,
    "debug": True
})

# 2. Create a ChatHistory instance and set session context
chat_history = ChatHistory()
chat_history.set_session_context(
    session_id="session-12345",
    agent_id="demo-agent-007",
    user_id="user-jane-doe",
    env="dev"
)

# 3. Add messages to the chat history with more properties
chat_history.add_user_message(
    "Hello, NexaMem!",
    tokens=5,
    data=json.dumps({"source": "notebook"})
)
chat_history.add_assistant_message(
    "Hello! How can I assist you today?",
    tokens=10
)

# 4. Retrieve and print the chat history, showing the new properties
history = chat_history.get_history()
for message in history:
    print(
        f"Role: {message['role']}, Content: \"{message['content']}\"\n"
        f"  Agent: {message['agent_id']}, User: {message['user_id']}, Session: {message['session_id']}\n"
        f"  Env: {message['env']}, Tokens: {message['tokens']}, Data: {message['data']}\n"
        f"  Timestamp: {message['timestamp']}\n"
    )

# 5. Clear the history for this session
# chat_history.clear()
# print("\nChat history cleared.")

[DEBUG] Initialized Redis connection: {'host': 'nexamem-test.redis.cache.windows.net', 'port': 6380, 'ssl': True, 'db': 0, 'connected': True, 'use_fakeredis': False, 'is_azure': False}
[DEBUG] Loaded 2 messages from Redis
[DEBUG] ChatHistory initialized with storage='redis'
[DEBUG] Set session context: {'agent_id': 'demo-agent-007', 'user_id': 'user-jane-doe', 'session_id': 'session-12345', 'env': 'dev'}
[DEBUG] Saved message to Redis: nexamem:chat_history:ed645440-3f60-4d6d-8f57-4f257b1c9563
[DEBUG] Added message: role='user' content='Hello, NexaMem!' agent_id='demo-agent-007' session_id='session-12345'
[DEBUG] Saved message to Redis: nexamem:chat_history:29ec25cf-6a2d-4233-a69f-ac93538fa0b8
[DEBUG] Added message: role='assistant' content='Hello! How can I assist you today?' agent_id='demo-agent-007' session_id='session-12345'
[DEBUG] Saved message to Redis: nexamem:chat_history:ed645440-3f60-4d6d-8f57-4f257b1c9563
[DEBUG] Added message: role='user' content='Hello, NexaMem!' agent_id=

### Using AIMemory with YAML Configuration

The `AIMemory` class provides a more advanced, policy-driven interface for managing conversational memory. It uses a YAML file to configure different memory "channels," each with its own TTL (Time To Live) and policies.

First, let's ensure a `channels.yaml` file exists.

In [12]:
import os
from nexamem.aimemory import AIMemory
from nexamem.memory_scope import MemoryScope

# Define the YAML configuration as a string (using correct dictionary format)
channels_yaml_content = """
channels:
  general:
    ttl_sec: 86400  # 24 hours
    encrypt: false

  customer_support:
    ttl_sec: 604800  # 7 days
    encrypt: false
"""

# Write the YAML content to a file
channels_yaml_path = "channels.yaml"
with open(channels_yaml_path, "w") as f:
    f.write(channels_yaml_content)

print(f"Created '{channels_yaml_path}' for AIMemory configuration.")

# 2. Initialize AIMemory with Redis config and channels file
# We need to unpack the redis_config dictionary into individual parameters
ai_memory = AIMemory(
    redis_host=redis_config["host"],
    redis_port=redis_config["port"],
    redis_password=redis_config["password"],
    redis_ssl=redis_config["ssl"],
    channels_yaml=channels_yaml_path
)

# 3. Define a memory scope for the interaction (including required agent_id)
scope = MemoryScope(
    agent_id="demo-agent-007",
    user_id="user-jane-doe",
    session_id="session-12345"
)

# 4. Write a message to the 'general' channel
print("\nWriting to 'general' channel...")
ai_memory.write(scope, "general", "This is a test message in the general channel.")

# 5. Read the memory from the 'general' channel
print("Reading from 'general' channel...")
messages = ai_memory.read(scope, "general")

# 6. Print the retrieved messages
for msg in messages:
    print(f"  - Retrieved: {msg}")

# 7. Clean up the created YAML file
# os.remove(channels_yaml_path)
# print(f"\nCleaned up '{channels_yaml_path}'.")

Created 'channels.yaml' for AIMemory configuration.

Writing to 'general' channel...
Reading from 'general' channel...

Writing to 'general' channel...
Reading from 'general' channel...
  - Retrieved: ['This is a test message in the general channel.', 'This is a test message in the general channel.']
  - Retrieved: {'total_messages': 2, 'total_bytes': 92, 'truncated': False}
  - Retrieved: ['This is a test message in the general channel.', 'This is a test message in the general channel.']
  - Retrieved: {'total_messages': 2, 'total_bytes': 92, 'truncated': False}


### Inspecting Redis Keys and Values

This section demonstrates how to directly inspect the keys and values stored in Redis by NexaMem. This is useful for debugging and understanding how data is stored in the cache.

In [13]:
import redis

# Create a direct Redis client connection using the same config
redis_client = redis.Redis(
    host=redis_config["host"],
    port=redis_config["port"],
    password=redis_config["password"],
    ssl=redis_config["ssl"],
    decode_responses=True  # This ensures we get string responses instead of bytes
)

print("=== Redis Keys and Values Inspection ===\n")

# 1. Get all keys in the Redis database
print("1. All keys in Redis:")
all_keys = redis_client.keys("*")
if all_keys:
    for key in sorted(all_keys):
        print(f"  - {key}")
else:
    print("  No keys found in Redis")

print(f"\nTotal keys: {len(all_keys)}\n")

# 2. Get values for each key and show their types
print("2. Key details and values:")
for key in sorted(all_keys):
    key_type = redis_client.type(key)
    ttl = redis_client.ttl(key)
    
    print(f"Key: {key}")
    print(f"  Type: {key_type}")
    print(f"  TTL: {ttl} seconds" + (" (no expiration)" if ttl == -1 else ""))
    
    try:
        if key_type == "string":
            value = redis_client.get(key)
            print(f"  Value: {value}")
        elif key_type == "list":
            value = redis_client.lrange(key, 0, -1)
            print(f"  Value (list): {value}")
        elif key_type == "hash":
            value = redis_client.hgetall(key)
            print(f"  Value (hash): {value}")
        elif key_type == "set":
            value = redis_client.smembers(key)
            print(f"  Value (set): {value}")
        elif key_type == "zset":
            value = redis_client.zrange(key, 0, -1, withscores=True)
            print(f"  Value (sorted set): {value}")
    except Exception as e:
        print(f"  Error reading value: {e}")
    
    print()

# 3. Show NexaMem-specific key patterns
print("3. NexaMem key analysis:")
nexamem_keys = [key for key in all_keys if "nexamem" in key.lower() or any(pattern in key for pattern in ["session", "agent", "user", "chat", "history"])]
if nexamem_keys:
    print("NexaMem-related keys:")
    for key in sorted(nexamem_keys):
        print(f"  - {key}")
else:
    print("No obvious NexaMem-related keys found")

# 4. Close the Redis connection
redis_client.close()
print("\nRedis inspection complete.")

=== Redis Keys and Values Inspection ===

1. All keys in Redis:
  - aimemory:audit
  - nexamem:chat_history:29ec25cf-6a2d-4233-a69f-ac93538fa0b8
  - nexamem:chat_history:b30b7d12-c2ac-4dc0-a76f-cf9bd716fa9f
  - nexamem:chat_history:d06bdb30-b867-42d4-9d7a-2908bf6d1458
  - nexamem:chat_history:ed645440-3f60-4d6d-8f57-4f257b1c9563
  - nexamem:chat_history:index
  - prod:demo-agent-007:user-jane-doe:session-12345:general:b6e808f1-93c2-41fc-84df-c7f7bae84aea
  - prod:demo-agent-007:user-jane-doe:session-12345:general:dced3fe2-42c8-4b41-90ce-b76c36f627da
  - prod:demo-agent-007:user-jane-doe:session-12345:general:index

Total keys: 9

2. Key details and values:
Key: aimemory:audit
  Type: stream
  TTL: 31536000 seconds

Key: nexamem:chat_history:29ec25cf-6a2d-4233-a69f-ac93538fa0b8
  Type: string
  TTL: 2591998 seconds
  - aimemory:audit
  - nexamem:chat_history:29ec25cf-6a2d-4233-a69f-ac93538fa0b8
  - nexamem:chat_history:b30b7d12-c2ac-4dc0-a76f-cf9bd716fa9f
  - nexamem:chat_history:d06bdb